# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and analyzing the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library. All entities are referenced by their `@id` fields according to the [Croissant](https://mlcommons.org/croissant/) standard.

### Dataset Source
The dataset source is provided via a Croissant schema URL and is FAIR-compliant.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the Croissant schema URL for the dataset
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Display basic metadata
print(f"Dataset Title: {dataset.metadata.name}")
print(f"Description:   {dataset.metadata.description}")
print(f"Published:     {dataset.metadata.datePublished}")
print(f"Version:       {dataset.metadata.version}")
print(f"License:       {dataset.metadata.license}")
print(f"Keywords:      {', '.join(dataset.metadata.keywords)}")

## 2. Data Overview
Review available record sets, their fields, and corresponding `@id`s.

In [ ]:
# List the RecordSets and their IDs (@id)
print("Available record sets (by @id):")
record_sets = dataset.metadata.recordSets
record_set_ids = []

for rset in record_sets:
    print(f"@id: {rset.id}    -- name: {getattr(rset, 'name', '<no name>')}")
    record_set_ids.append(rset.id)
    if hasattr(rset, "fields"):
        print("    Fields:")
        for field in rset.fields:
            print(f"       @id: {field.id}    -- name: {getattr(field, 'name', '<no name>')}; type: {getattr(field, 'dataType', '<unknown>')}")
    else:
        print("    No fields described.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame with field `@id`s as columns.

In [ ]:
# Choose the first available record set for demonstration
if not record_set_ids:
    raise Exception("No record sets available in the Croissant metadata.")

main_record_set_id = record_set_ids[0]
print(f"Example record set @id: {main_record_set_id}")

# List all available record sets for reference
print("All record set @ids:")
for rsid in record_set_ids:
    print(f"  - {rsid}")

dataframes = {}
for rsid in record_set_ids:
    # Load records from this record set @id
    records = list(dataset.records(record_set=rsid))
    if records:
        # Store in a DataFrame (columns are field @ids)
        dataframes[rsid] = pd.DataFrame(records)
    else:
        dataframes[rsid] = pd.DataFrame()

# Display columns for the main record set
print(f"Columns for record set '{main_record_set_id}':\n", dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Let's process the dataset: select a numeric field (by its `@id`), filter, normalize, and group the data.

In [ ]:
# Use the first numeric field found for demonstration
main_df = dataframes[main_record_set_id]

# Detect numeric fields by looking for float/int dtypes or sample values
numeric_field_id = None
for col in main_df.columns:
    try:
        # Try converting to numeric; if a reasonable number of non-nans, pick this field
        series = pd.to_numeric(main_df[col], errors='coerce')
        if series.notnull().sum() > 0:
            numeric_field_id = col
            break
    except Exception:
        continue
if numeric_field_id is None:
    raise Exception("No numeric field found in the main record set.")

print(f"Selected numeric field for analysis: {numeric_field_id}")

# Ensure numeric type and drop missing values
main_df[numeric_field_id] = pd.to_numeric(main_df[numeric_field_id], errors='coerce')
df_numeric = main_df.dropna(subset=[numeric_field_id]).copy()

# Set threshold at the median for demonstration
threshold = df_numeric[numeric_field_id].median()
filtered_df = df_numeric[df_numeric[numeric_field_id] > threshold].copy()
print(f"Filtered records where {numeric_field_id} > {threshold:.2f} (median):")
print(filtered_df[[numeric_field_id]].head())

# Normalization (z-score)
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Attempt to group by a categorical field (first non-numeric found)
group_field_id = None
for col in main_df.columns:
    if col == numeric_field_id:
        continue
    # If column type is object and not float/int, use as group
    if main_df[col].dtype == object:
        group_field_id = col
        break

if group_field_id and group_field_id in filtered_df.columns:
    # Group by this field and calculate mean
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
    print(grouped_df.head())
else:
    print("No suitable categorical field for grouping found (by heuristic).")

## 5. Visualization
Visualize distribution of the selected numeric field and its relationship with the group field (if found).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid')

# Distribution histogram
plt.figure(figsize=(8,4))
sns.histplot(df_numeric[numeric_field_id], bins=30, kde=True, color='skyblue')
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Frequency")
plt.show()

# Boxplot grouped by categorical field, if found
if group_field_id and group_field_id in df_numeric.columns:
    n_unique = df_numeric[group_field_id].nunique()
    if n_unique < 30:  # Only if not too many groups
        plt.figure(figsize=(10,4))
        sns.boxplot(data=df_numeric, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} grouped by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
- The dataset provides mass spectrometry protein records with detailed fields, accessible using FAIR and Croissant conventions.
- We loaded all record sets using their `@id` and demonstrated typical preprocessing: filtering, normalization, and grouping by categorical fields.
- Visualization explored distributions and possible group structure in the numeric data.
- For further work: feature selection, imputation, more advanced visualizations, or modeling.

Refer to the Croissant schema for rich FAIR metadata and precise field definitions.